# Model 2: Aspect-Based Sentiment Analysis & Intent Classification

Uses two HuggingFace datasets instead of Reviews.csv:

| Dataset | Purpose |
|---|---|
| [NEUDM/mams](https://huggingface.co/datasets/NEUDM/mams) | Supervised ABSA training (aspect-term + aspect-category sentiment) |
| [Multilingual-NLP/M-ABSA](https://huggingface.co/datasets/Multilingual-NLP/M-ABSA) | Additional ABSA evaluation / cross-domain test |

Outputs:
- `absa_model.rds` — glmnet ABSA sentiment classifier
- `absa_vocab.rds` — vocabulary for the ABSA model
- `intent_model.rds` — glmnet intent classifier (trained on MAMS labels)
- `intent_vocab.rds` — vocabulary for the intent model

## Step 1: Install / Load Packages

In [133]:
required_r <- c('tm', 'glmnet', 'Matrix', 'reticulate')
for (pkg in required_r) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    install.packages(pkg, repos = 'https://cloud.r-project.org')
  }
}
library(tm); library(Matrix); library(glmnet); library(reticulate)

required_py <- c('datasets', 'pandas')
missing_py <- required_py[!vapply(required_py, py_module_available, logical(1))]
if (length(missing_py)) {
  py_install(missing_py, pip = TRUE)
}

cat(sprintf('Reticulate Python: %s\n', py_config()$python))
ds <- import('datasets')
pd <- import('pandas')
cat('All packages ready\n')

Reticulate Python: /Users/ahmedtahirshekhani/Library/Caches/org.R-project.R/R/reticulate/uv/cache/archive-v0/f6djvnGBZ62cqtrq/bin/python
All packages ready


## Step 2: Load MAMS Dataset

MAMS (Multi-Aspect Multi-Sentiment) has two sub-tasks:
- **ATSA** — Aspect-Term Sentiment Analysis (fine-grained, per term)
- **ACSA** — Aspect-Category Sentiment Analysis (coarser, per category)

We load ACSA for training (cleaner categories) and inspect ATSA for reference.

In [134]:
# Load with default config — no sub-config available
mams <- ds$load_dataset('NEUDM/mams')

cat('=== MAMS splits ===\n')
print(py_to_r(mams$`__repr__`()))

cat('\n=== Column names ===\n')
print(py_to_r(mams$train$column_names))

cat('\n=== First example ===\n')
ex <- mams$train[0L]
for (nm in names(ex)) {
  val <- tryCatch(paste(py_to_r(ex[[nm]]), collapse = ' | '), error = function(e) '(complex type)')
  cat(sprintf('  %-20s: %s\n', nm, substr(val, 1, 120)))
}

=== MAMS splits ===
[1] "DatasetDict({\n    train: Dataset({\n        features: ['task_type', 'dataset', 'input', 'output', 'situation', 'label', 'extra', 'instruction'],\n        num_rows: 7446\n    })\n    validation: Dataset({\n        features: ['task_type', 'dataset', 'input', 'output', 'situation', 'label', 'extra', 'instruction'],\n        num_rows: 900\n    })\n    test: Dataset({\n        features: ['task_type', 'dataset', 'input', 'output', 'situation', 'label', 'extra', 'instruction'],\n        num_rows: 900\n    })\n})"

=== Column names ===
[1] "task_type"   "dataset"     "input"       "output"      "situation"  
[6] "label"       "extra"       "instruction"

=== First example ===
  task_type           : generation
  dataset             : mams
  input               : It might be the best sit down food I've had in the area, so if you are going to the upright citizen brigade, or the gard
  output              : [['food', 'positive'], ['place', 'neutral']]
  situation        

In [135]:
# Convert splits to R — use 'mams' (default config)
acsa_train <- as.data.frame(py_to_r(mams$train$to_pandas()), stringsAsFactors = FALSE)
acsa_val   <- as.data.frame(py_to_r(mams$validation$to_pandas()), stringsAsFactors = FALSE)
acsa_test  <- as.data.frame(py_to_r(mams$test$to_pandas()), stringsAsFactors = FALSE)
rownames(acsa_train) <- NULL
rownames(acsa_val)   <- NULL
rownames(acsa_test)  <- NULL

cat(sprintf('Train: %d | Val: %d | Test: %d\n',
            nrow(acsa_train), nrow(acsa_val), nrow(acsa_test)))
cat('\nAll columns:\n'); print(names(acsa_train))
cat('\nSample row:\n'); print(acsa_train[1, ])

Train: 7446 | Val: 900 | Test: 900

All columns:
[1] "task_type"   "dataset"     "input"       "output"      "situation"  
[6] "label"       "extra"       "instruction"

Sample row:
   task_type dataset
1 generation    mams
                                                                                                                                                            input
1 It might be the best sit down food I've had in the area, so if you are going to the upright citizen brigade, or the garden, it could be just the place for you.
                                                 output situation label extra
1 [['staff', 'positive'], ['miscellaneous', 'neutral']]      none            
                                                                                                                                                                                                                                                                                                        

## Step 3: Explore & Clean MAMS-ACSA

In [136]:
# Extract only the two needed columns as plain R character vectors.
# Convert the split row-by-row so we never touch the nested list columns as a whole in R.
extract_cols <- function(split) {
  rows <- py_to_r(split$to_list())

  data.frame(
    raw = vapply(rows, function(row) paste(unlist(row$input), collapse = ' '), character(1)),
    output = vapply(rows, function(row) paste(as.character(row$output), collapse = ' '), character(1)),
    stringsAsFactors = FALSE
  )
}

acsa_raw <- rbind(
  extract_cols(mams$train),
  extract_cols(mams$validation),
  extract_cols(mams$test)
 )
rownames(acsa_raw) <- NULL

# Parse bracketed tuples from the output string.
# If nothing matches, the parser stops with the first output sample so the mismatch is visible.
parse_acsa <- function(df) {
  rows <- lapply(seq_len(nrow(df)), function(i) {
    raw_text <- df$raw[i]
    out_str  <- df$output[i]

    tuples <- regmatches(out_str, gregexpr("\\[[^\\]]+\\]", out_str, perl = TRUE))[[1]]
    if (length(tuples) == 0) return(NULL)

    do.call(rbind, lapply(tuples, function(tuple) {
      parts <- gsub("'", "", regmatches(tuple, gregexpr("'([^']+)'", tuple, perl = TRUE))[[1]])
      if (length(parts) >= 4) {
        category <- parts[2]
        polarity <- parts[3]
      } else if (length(parts) >= 2) {
        category <- parts[length(parts) - 1]
        polarity <- parts[length(parts)]
      } else {
        return(NULL)
      }
      data.frame(raw = raw_text, category = category, polarity = polarity,
                 stringsAsFactors = FALSE)
    }))
  })

  rows <- Filter(Negate(is.null), rows)
  if (!length(rows)) {
    first_output <- if (nrow(df) > 0) df$output[1] else '<empty>'
    stop(sprintf('No ACSA rows parsed. First output sample: %s', first_output))
  }

  do.call(rbind, rows)
}

acsa_all <- parse_acsa(acsa_raw)
rownames(acsa_all) <- NULL

# Keep only the three main sentiment classes; drop 'conflict'
acsa_all <- acsa_all[acsa_all$polarity %in% c('positive','negative','neutral'), ]
acsa_all$polarity <- factor(acsa_all$polarity,
                            levels = c('negative','neutral','positive'))

cat(sprintf('Rows after dropping conflict: %d\n', nrow(acsa_all)))
cat('\nCategory distribution:\n')
print(sort(table(acsa_all$category), decreasing = TRUE))
cat('\nPolarity distribution:\n')
print(table(acsa_all$polarity))

Rows after dropping conflict: 22726

Category distribution:

                                                      food 
                                                      3697 
                                                     staff 
                                                      1849 
                                                   service 
                                                      1253 
                                             miscellaneous 
                                                      1219 
                                                      menu 
                                                      1103 
                                                     place 
                                                      1018 
                                                     price 
                                                       528 
                                                    waiter 
                                       

In [137]:
# Quick data samples after parsing
cat('=== Parsed ACSA samples ===\n')
print(utils::head(acsa_all, 8))

cat('\n=== Raw split samples ===\n')
print(utils::head(acsa_raw, 3))

cat('\n=== Category / polarity summary ===\n')
print(sort(table(acsa_all$category), decreasing = TRUE))
print(table(acsa_all$polarity))

=== Parsed ACSA samples ===
                                                                                                                                                              raw
1 It might be the best sit down food I've had in the area, so if you are going to the upright citizen brigade, or the garden, it could be just the place for you.
2 It might be the best sit down food I've had in the area, so if you are going to the upright citizen brigade, or the garden, it could be just the place for you.
3                                                                          Hostess was extremely accommodating when we arrived an hour early for our reservation.
4                                                                          Hostess was extremely accommodating when we arrived an hour early for our reservation.
5                 We were a couple of minutes late for our reservation and minus one guest, but we didn't think we deserved the attitude we got from the hostess.


## Step 4: Feature Engineering for ABSA

Concatenate the review text with its aspect category so the model learns
which words matter for each specific aspect:

```
input = "The food was great but service was slow  ASPECT_FOOD"
label = positive
```

In [138]:
# Build input: sentence + aspect marker
acsa_all$input_text <- paste(acsa_all$raw,
                             paste0('ASPECT_', toupper(gsub(' ', '_', acsa_all$category))))

set.seed(42)
if (is.null(acsa_all) || !is.data.frame(acsa_all) || nrow(acsa_all) < 2L) {
  stop('Parsed ACSA data is empty or invalid; check the previous parsing cell.')
}
n <- as.integer(nrow(acsa_all))
train_size <- max(1L, floor(0.8 * n))
if (train_size >= n) train_size <- n - 1L
tr_idx <- sample.int(n, size = train_size)
absa_train <- acsa_all[tr_idx, ]
absa_test  <- acsa_all[-tr_idx, ]

cat(sprintf('Train: %d | Test: %d\n', nrow(absa_train), nrow(absa_test)))
cat('\nTrain polarity split:\n'); print(table(absa_train$polarity))

Train: 18180 | Test: 4546

Train polarity split:

negative  neutral positive 
    4818     8090     5272 


## Dataset Preview (Pandas-Style)

In [139]:
if (!requireNamespace('knitr', quietly = TRUE)) install.packages('knitr', repos = 'https://cloud.r-project.org')
library(knitr)

# Columns to display
preview_cols <- c('raw', 'category', 'polarity', 'input_text')

cat(sprintf('=== ABSA Train: %d rows x %d cols ===\n', nrow(absa_train), ncol(absa_train)))
cat(sprintf('=== ABSA Test : %d rows x %d cols ===\n\n', nrow(absa_test),  ncol(absa_test)))

# Helper: truncate long strings for display
trunc_str <- function(x, n = 80) ifelse(nchar(x) > n, paste0(substr(x, 1, n), '…'), x)

make_preview <- function(df, n = 8) {
  d <- utils::head(df[, preview_cols], n)
  d$raw        <- trunc_str(d$raw)
  d$input_text <- trunc_str(d$input_text, 60)
  d
}

cat('--- Train sample (first 8 rows) ---\n')
kable(make_preview(absa_train), format = 'simple', row.names = TRUE)


=== ABSA Train: 18180 rows x 4 cols ===
=== ABSA Test : 4546 rows x 4 cols ===

--- Train sample (first 8 rows) ---




        raw                                                                                        category              polarity   input_text                                                           
------  -----------------------------------------------------------------------------------------  --------------------  ---------  ---------------------------------------------------------------------
18753   If you've got the money and don't mind being packed in like sardines - then cozy<U+2026>   miscellaneous         positive   If you've got the money and don't mind being packed in like <U+2026> 
21657   Start with fennel-fragrant grilled sardines and the extraordinary pastas, such a<U+2026>   swiss chard-ricotta   positive   Start with fennel-fragrant grilled sardines and the extraord<U+2026> 
9290    What we were treated to, instead, was snooty attitude by the waiter and the mait<U+2026>   maitre                negative   What we were treated to, instead, was snooty attitude by t

In [140]:
cat('--- Test sample (first 8 rows) ---\n')
kable(make_preview(absa_test), format = 'simple', row.names = TRUE)

cat('\n--- Train polarity distribution ---\n')
kable(as.data.frame(table(Polarity = absa_train$polarity)), format = 'simple', row.names = FALSE)

cat('\n--- Test polarity distribution ---\n')
kable(as.data.frame(table(Polarity = absa_test$polarity)), format = 'simple', row.names = FALSE)

cat('\n--- Train category distribution (top 10) ---\n')
top_cats <- sort(table(absa_train$category), decreasing = TRUE)[1:10]
kable(as.data.frame(top_cats, responseName = 'Count'), col.names = c('Category', 'Count'), format = 'simple', row.names = FALSE)


--- Test sample (first 8 rows) ---




     raw                                                                                        category        polarity   input_text                                                           
---  -----------------------------------------------------------------------------------------  --------------  ---------  ---------------------------------------------------------------------
5    We were a couple of minutes late for our reservation and minus one guest, but we<U+2026>   miscellaneous   neutral    We were a couple of minutes late for our reservation and min<U+2026> 
14   The bill was surprisingly inexpensive considering we each had appetizers, an ent<U+2026>   price           positive   The bill was surprisingly inexpensive considering we each ha<U+2026> 
21   After ordering drinks, we both decided on the Paella Vallenciana, brought out on<U+2026>   miscellaneous   positive   After ordering drinks, we both decided on the Paella Vallenc<U+2026> 
30   After reading other reviews 


--- Train polarity distribution ---




Polarity    Freq
---------  -----
negative    4818
neutral     8090
positive    5272


--- Test polarity distribution ---




Polarity    Freq
---------  -----
negative    1206
neutral     2016
positive    1324


--- Train category distribution (top 10) ---




Category         Count
--------------  ------
food              2924
staff             1481
service            987
miscellaneous      982
menu               884
place              819
price              419
waiter             407
ambience           345
bar                294

## Step 5: Build Document-Term Matrices

In [141]:
clean_text <- function(x) {
  corpus <- VCorpus(VectorSource(as.character(x)))
  corpus <- tm_map(corpus, content_transformer(tolower))
  corpus <- tm_map(corpus, removePunctuation)
  corpus <- tm_map(corpus, removeNumbers)
  corpus <- tm_map(corpus, removeWords, stopwords('en'))
  corpus <- tm_map(corpus, stripWhitespace)
  corpus
}

train_corpus <- clean_text(absa_train$input_text)
test_corpus  <- clean_text(absa_test$input_text)

train_dtm <- DocumentTermMatrix(train_corpus,
  control = list(wordLengths = c(3, Inf), bounds = list(global = c(2L, Inf))))
test_dtm  <- DocumentTermMatrix(test_corpus,
  control = list(dictionary = Terms(train_dtm)))

x_train <- Matrix::sparseMatrix(i=train_dtm$i, j=train_dtm$j, x=as.numeric(train_dtm$v),
  dims=dim(train_dtm), dimnames=dimnames(train_dtm))
x_test  <- Matrix::sparseMatrix(i=test_dtm$i,  j=test_dtm$j,  x=as.numeric(test_dtm$v),
  dims=dim(test_dtm),  dimnames=dimnames(test_dtm))
y_train <- absa_train$polarity
y_test  <- absa_test$polarity

cat(sprintf('Train: %d x %d | Test: %d x %d\n',
            nrow(x_train), ncol(x_train), nrow(x_test), ncol(x_test)))

Train: 18180 x 9227 | Test: 4546 x 9227


## Step 6: Train ABSA Model (cv.glmnet)

In [142]:
set.seed(42)
absa_cv <- cv.glmnet(
  x_train, y_train,
  family       = 'multinomial',
  type.measure = 'class',
  nfolds       = 5
)
cat(sprintf('Best lambda : %.6f\n', absa_cv$lambda.min))
cat(sprintf('CV error    : %.4f\n', min(absa_cv$cvm)))

Best lambda : 0.005485
CV error    : 0.3805


## Step 7: Evaluate ABSA Model

In [143]:
pred_absa <- predict(absa_cv, newx = x_test, s = 'lambda.min', type = 'class')
pred_absa <- factor(as.character(pred_absa), levels = levels(y_test))

cm    <- table(Actual = y_test, Predicted = pred_absa)
total <- sum(cm)
cat('Confusion Matrix:\n'); print(cm)

classes <- levels(y_test)
metrics <- do.call(rbind, lapply(classes, function(cls) {
  TP <- cm[cls,cls]; FP <- sum(cm[,cls])-TP; FN <- sum(cm[cls,])-TP
  prec <- ifelse((TP+FP)==0, NA, TP/(TP+FP))
  rec  <- ifelse((TP+FN)==0, NA, TP/(TP+FN))
  f1   <- ifelse(is.na(prec)|is.na(rec)|(prec+rec)==0, NA, 2*prec*rec/(prec+rec))
  data.frame(Class=cls, Precision=round(prec,4), Recall=round(rec,4),
             F1=round(f1,4), Support=as.integer(table(y_test)[cls]))
}))
rownames(metrics) <- NULL
cat('\nPer-Class Metrics:\n'); print(metrics)

support     <- metrics$Support
macro_f1    <- mean(metrics$F1, na.rm=TRUE)
weighted_f1 <- sum(metrics$F1 * support, na.rm=TRUE) / sum(support)
cat(sprintf('\nAccuracy    : %.4f\n', sum(diag(cm))/total))
cat(sprintf('Macro F1    : %.4f\n', macro_f1))
cat(sprintf('Weighted F1 : %.4f\n', weighted_f1))

Confusion Matrix:
          Predicted
Actual     negative neutral positive
  negative      669     331      206
  neutral       141    1608      267
  positive      188     586      550

Per-Class Metrics:
     Class Precision Recall     F1 Support
1 negative    0.6703 0.5547 0.6071    1206
2  neutral    0.6368 0.7976 0.7082    2016
3 positive    0.5376 0.4154 0.4687    1324

Accuracy    : 0.6219
Macro F1    : 0.5947
Weighted F1 : 0.6116


## Step 8: Save ABSA Model

In [144]:
saveRDS(absa_cv,        'absa_model.rds')
saveRDS(Terms(train_dtm),'absa_vocab.rds')
cat(sprintf('absa_model.rds saved  (lambda.min = %.6f)\n', absa_cv$lambda.min))
cat(sprintf('absa_vocab.rds saved  (%d terms)\n', ncol(x_train)))

absa_model.rds saved  (lambda.min = 0.005485)
absa_vocab.rds saved  (9227 terms)


## Step 9: Load M-ABSA and Evaluate Cross-Domain

Test whether the MAMS-trained model generalises to the M-ABSA dataset.

In [145]:
mabsa_raw <- ds$load_dataset('Multilingual-NLP/M-ABSA')

cat('=== M-ABSA splits ===\n')
print(py_to_r(mabsa_raw$`__repr__`()))
cat('\n=== M-ABSA column names ===\n')
print(py_to_r(mabsa_raw$train$column_names))

cat('\n=== First M-ABSA example ===\n')
ex2 <- mabsa_raw$train[0L]
for (nm in names(ex2)) cat(sprintf('  %s: %s\n', nm, paste(py_to_r(ex2[[nm]]), collapse=' | ')))

=== M-ABSA splits ===
[1] "DatasetDict({\n    train: Dataset({\n        features: ['text'],\n        num_rows: 184716\n    })\n    validation: Dataset({\n        features: ['text'],\n        num_rows: 45906\n    })\n    test: Dataset({\n        features: ['text'],\n        num_rows: 79674\n    })\n})"

=== M-ABSA column names ===
[1] "text"

=== First M-ABSA example ===
  text: <U+200E><U+0644><U+0643><U+0646><U+0646><U+064A> <U+0634><U+0639><U+0631><U+062A> <U+062F><U+0627><U+0626><U+0645><U+064B><U+0627> <U+0623><U+0646><U+0646><U+064A> <U+0623><U+0641><U+062A><U+0642><U+062F> <U+0627><U+0644><U+0645><U+0639><U+0644><U+0648><U+0645><U+0627><U+062A> <U+0627><U+0644><U+0646><U+0638><U+0631><U+064A><U+0629>.####[]


In [146]:
mabsa_df <- as.data.frame(py_to_r(mabsa_raw$train$to_pandas()), stringsAsFactors = FALSE)
rownames(mabsa_df) <- NULL
cat('M-ABSA columns:\n'); print(names(mabsa_df))
cat('\nRows:', nrow(mabsa_df), '\n')
str(mabsa_df[1:3, ])

M-ABSA columns:
[1] "text"

Rows: 184716 
<ArrowStringArray>
['<U+200E><U+0648><U+0623><U+062E><U+064A><U+0631><U+064B><U+0627><U+060C> <U+0645><U+0642><U+062F><U+0645><U+0629> <U+0642><U+0635><U+064A><U+0631><U+0629> <U+0639><U+0646> <U+0641><U+0646> <U+0627><U+0633><U+062A><U+062E><U+062F><U+0627><U+0645> <U+0623><U+062D><U+0643><U+0627><U+0645> <U+0627><U+0644><U+0642><U+0636><U+0627><U+0621> <U+0643><U+062D><U+062F><U+062B> <U+0623><U+0648> <U+0627><U+062E><U+062A><U+0628><U+0627><U+0631><U+060C> <U+0639><U+0644><U+0649> <U+0633><U+0628><U+064A><U+0644> <U+0627><U+0644><U+0645><U+062B><U+0627><U+0644><U+061B> <U+0641><U+064A> "<U+0645><U+064A><U+0631><U+0627><U+0646><U+062F><U+0627>"<U+060C> <U+0623><U+0648> "<U+0643><U+064A><U+0644><U+0631>"<U+060C> <U+0623><U+0648> "<U+0628><U+0631><U+0627><U+0648><U+0646>"<U+060C> <U+0623><U+0648> "<U+0627><U+062E><U+062A><U+0628><U+0627><U+0631> <U+0645><U+0627><U+0643><U+062F><U+0648><U+0646><U+064A><U+0644> <U+062F><U+0648><U+063A><U+0644><U+

In [147]:
# mabsa_df$text is Arrow-backed and stays as a Python environment in R.
# Re-extract as a plain R character vector using the same to_list() path as c04.
raw_texts <- vapply(
  py_to_r(mabsa_raw$train$to_list()),
  function(row) as.character(row[["text"]]),
  character(1)
)
cat(sprintf('Rows extracted: %d\n', length(raw_texts)))
cat('First entry (truncated):\n')
cat(substr(raw_texts[3], 1, 200), '\n')  # row 3 has annotations

# M-ABSA packs everything into one "text" column:
#   <review sentence>####[['aspect_term', 'category', 'polarity'], ...]
parse_mabsa <- function(raw_vec) {
  rows <- lapply(raw_vec, function(entry) {
    parts     <- strsplit(entry, '####', fixed = TRUE)[[1]]
    if (length(parts) < 2) return(NULL)
    review    <- trimws(parts[1])
    annot_str <- trimws(parts[2])
    if (annot_str %in% c('[]', '', 'NULL', 'null')) return(NULL)

    tuples <- regmatches(annot_str,
      gregexpr("\\[[^\\[\\]]+\\]", annot_str, perl = TRUE))[[1]]
    if (!length(tuples)) return(NULL)

    do.call(rbind, lapply(tuples, function(tup) {
      tokens  <- gsub("^'|'$", '',
        regmatches(tup, gregexpr("'([^']*)'", tup, perl = TRUE))[[1]])
      if (length(tokens) < 3) return(NULL)
      category <- tokens[2]
      polarity <- tolower(tokens[3])
      if (polarity %in% c('positive','negative','neutral') && category != 'NULL')
        data.frame(raw = review, category = category,
                   polarity = polarity, stringsAsFactors = FALSE)
      else NULL
    }))
  })
  rows <- Filter(Negate(is.null), rows)
  if (!length(rows)) stop('No M-ABSA rows parsed.')
  result <- do.call(rbind, rows)
  rownames(result) <- NULL
  result
}

mabsa_clean <- parse_mabsa(raw_texts)
mabsa_clean$polarity   <- factor(mabsa_clean$polarity,
                                  levels = c('negative','neutral','positive'))
mabsa_clean$input_text <- paste(mabsa_clean$raw,
  paste0('ASPECT_', toupper(gsub(' ','_', mabsa_clean$category))))

cat(sprintf('\nM-ABSA rows (clean): %d\n', nrow(mabsa_clean)))
cat('\nPolarity distribution:\n'); print(table(mabsa_clean$polarity))
cat('\nTop categories:\n')
print(sort(table(mabsa_clean$category), decreasing = TRUE)[1:10])
cat('\nSample rows:\n')
print(utils::head(mabsa_clean[, c('raw','category','polarity')], 5))


Rows extracted: 184716
First entry (truncated):
<U+200E><U+0633><U+0623><U+0639><U+0648><U+062F> <U+0628><U+0627><U+0644><U+062A><U+0623><U+0643><U+064A><U+062F> <U+0625><U+0644><U+0649> <U+0628><U+0639><U+0636> <U+0623><U+062C><U+0632><U+0627><U+0621> <U+0647><U+0630><U+0647> <U+0627><U+0644><U+062F><U+0648><U+0631><U+0629> <U+0641><U+064A> <U+0627><U+0644><U+0645><U+0633><U+062A><U+0642><U+0628><U+0644>.####[] 

M-ABSA rows (clean): 220139

Polarity distribution:

negative  neutral positive 
   50195     8805   161139 

Top categories:

      food quality    service general    Overall#Overall     course general 
             22375              13394              11428              10099 
      food general      hotel general     LAPTOP#GENERAL restaurant general 
              9491               8197               7649               6380 
  location general    Logistics#Speed 
              5088               5021 

Sample rows:
                                                       

In [148]:
# Build DTM for M-ABSA using the ABSA model vocabulary
absa_vocab   <- readRDS('absa_vocab.rds')
mabsa_corpus <- clean_text(mabsa_clean$input_text)
mabsa_dtm    <- DocumentTermMatrix(mabsa_corpus, control=list(dictionary=absa_vocab))
x_mabsa <- Matrix::sparseMatrix(i=mabsa_dtm$i, j=mabsa_dtm$j, x=as.numeric(mabsa_dtm$v),
  dims=dim(mabsa_dtm), dimnames=dimnames(mabsa_dtm))
y_mabsa <- mabsa_clean$polarity

pred_mabsa <- predict(absa_cv, newx=x_mabsa, s='lambda.min', type='class')
pred_mabsa <- factor(as.character(pred_mabsa), levels=levels(y_mabsa))

cm_m    <- table(Actual=y_mabsa, Predicted=pred_mabsa)
acc_m   <- sum(diag(cm_m)) / sum(cm_m)
cat('=== Cross-Domain Evaluation (M-ABSA) ===\n')
print(cm_m)
cat(sprintf('\nAccuracy on M-ABSA: %.4f\n', acc_m))

=== Cross-Domain Evaluation (M-ABSA) ===
          Predicted
Actual     negative neutral positive
  negative      193   49399      603
  neutral        10    8669      126
  positive       63  156494     4582

Accuracy on M-ABSA: 0.0611


## Step 10: Intent Classification

MAMS does not have intent labels, so we derive them from the ACSA data using
the same heuristic rules as before, then train a glmnet classifier.

In [149]:
create_intent_label <- function(text, polarity) {
  t <- tolower(as.character(text))
  if (grepl('\\?', t) ||
      grepl('^(what|how|when|where|why|who|is |are |does |do |can |could |will |would )', t))
    return('inquiry')
  if (grepl('\\b(should|suggest|recommendation|please|consider|improve|wish|would be better|maybe|perhaps|add |include|hope|next time)\\b', t))
    return('suggestion')
  if (polarity == 'negative' &&
      grepl('\\b(broken|defective|damaged|refund|return|failed|horrible|terrible|awful|worst|misleading|never again)\\b', t))
    return('complaint')
  if (polarity == 'negative')
    return('complaint')
  if (polarity == 'positive' &&
      grepl('\\b(amazing|excellent|perfect|fantastic|wonderful|love|recommend|best|awesome|outstanding|superb|great|happy|impressed)\\b', t))
    return('compliment')
  'general'
}

acsa_all$intent <- mapply(create_intent_label,
                          acsa_all$raw, as.character(acsa_all$polarity),
                          SIMPLIFY = TRUE)
acsa_all$intent <- factor(acsa_all$intent,
  levels = c('complaint','compliment','general','inquiry','suggestion'))

cat('Intent distribution:\n'); print(table(acsa_all$intent))

Intent distribution:

 complaint compliment    general    inquiry suggestion 
      5456       1432      13980        987        871 


In [150]:
set.seed(42)
n2       <- nrow(acsa_all)
tr2_idx  <- sample(seq_len(n2), size = floor(0.8 * n2))
int_train <- acsa_all[ tr2_idx, ]
int_test  <- acsa_all[-tr2_idx, ]

# Use raw sentence (not the aspect-augmented text) for intent
tc_train <- clean_text(int_train$raw)
tc_test  <- clean_text(int_test$raw)

int_train_dtm <- DocumentTermMatrix(tc_train,
  control=list(wordLengths=c(3,Inf), bounds=list(global=c(2L,Inf))))
int_test_dtm  <- DocumentTermMatrix(tc_test,
  control=list(dictionary=Terms(int_train_dtm)))

xi_train <- Matrix::sparseMatrix(i=int_train_dtm$i, j=int_train_dtm$j,
  x=as.numeric(int_train_dtm$v), dims=dim(int_train_dtm), dimnames=dimnames(int_train_dtm))
xi_test  <- Matrix::sparseMatrix(i=int_test_dtm$i,  j=int_test_dtm$j,
  x=as.numeric(int_test_dtm$v),  dims=dim(int_test_dtm),  dimnames=dimnames(int_test_dtm))
yi_train <- int_train$intent
yi_test  <- int_test$intent

cat(sprintf('Intent Train: %d | Test: %d | Vocab: %d\n',
            nrow(xi_train), nrow(xi_test), ncol(xi_train)))

Intent Train: 18180 | Test: 4546 | Vocab: 8685


In [151]:
set.seed(42)
intent_cv <- cv.glmnet(xi_train, yi_train,
  family='multinomial', type.measure='class', nfolds=5)

pred_int <- predict(intent_cv, newx=xi_test, s='lambda.min', type='class')
pred_int <- factor(as.character(pred_int), levels=levels(yi_test))
cm_i     <- table(Actual=yi_test, Predicted=pred_int)
cat('Intent Confusion Matrix:\n'); print(cm_i)
cat(sprintf('\nIntent Accuracy: %.4f\n', sum(diag(cm_i))/sum(cm_i)))

Warning message:
"from glmnet C++ code (error code -23); Convergence for 23th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -24); Convergence for 24th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -24); Convergence for 24th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -28); Convergence for 28th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -22); Convergence for 22th lambda value not reached after maxit=100000 iterations; solutions for larger lambdas returned"
Warning message:
"from glmnet C++ code (error code -24); Convergence for 24th lambda value not reached after maxit=100000 iterations; s

Intent Confusion Matrix:
            Predicted
Actual       complaint compliment general inquiry suggestion
  complaint          7         13    1054      11          0
  compliment         2         21     256       1          0
  general            3         16    2768       8          0
  inquiry            0          2     165      33          7
  suggestion         0          1      73       0        105

Intent Accuracy: 0.6454


In [152]:
saveRDS(intent_cv,             'intent_model.rds')
saveRDS(Terms(int_train_dtm),  'intent_vocab.rds')
cat(sprintf('intent_model.rds saved  (lambda.min = %.6f)\n', intent_cv$lambda.min))
cat(sprintf('intent_vocab.rds saved  (%d terms)\n', ncol(xi_train)))

intent_model.rds saved  (lambda.min = 0.013309)
intent_vocab.rds saved  (8685 terms)


## Step 11: Full Enriched Output — All Models Together

In [153]:
# Load saved models
absa_model_loaded   <- suppressWarnings(readRDS('absa_model.rds'))
absa_vocab_loaded   <- readRDS('absa_vocab.rds')
intent_model_loaded <- suppressWarnings(readRDS('intent_model.rds'))
intent_vocab_loaded <- readRDS('intent_vocab.rds')

predict_aspect <- function(text, aspect_category) {
  input  <- paste(text, paste0('ASPECT_', toupper(gsub(' ','_', aspect_category))))
  corpus <- clean_text(input)
  dtm    <- DocumentTermMatrix(corpus, control=list(dictionary=absa_vocab_loaded))
  x      <- Matrix::sparseMatrix(i=dtm$i, j=dtm$j, x=as.numeric(dtm$v),
                                 dims=dim(dtm), dimnames=dimnames(dtm))
  as.character(predict(absa_model_loaded, newx=x, s='lambda.min', type='class'))
}

predict_intent_ml <- function(texts) {
  corpus <- clean_text(texts)
  dtm    <- DocumentTermMatrix(corpus, control=list(dictionary=intent_vocab_loaded))
  x      <- Matrix::sparseMatrix(i=dtm$i, j=dtm$j, x=as.numeric(dtm$v),
                                 dims=dim(dtm), dimnames=dimnames(dtm))
  as.character(predict(intent_model_loaded, newx=x, s='lambda.min', type='class'))
}

test_cases <- list(
  list(text='The food was absolutely delicious but the service was shockingly rude.',
       aspects=c('food','service')),
  list(text='Great value for money! Arrived quickly and packaging was perfect.',
       aspects=c('price','shipping','packaging')),
  list(text='I wish the portion size was bigger. The taste is decent though.',
       aspects=c('food','service')),
  list(text='Does this come in a larger size? The quality looks great in the photos.',
       aspects=c('food','ambience'))
)

cat('=== Full Enriched Output ===\n\n')
for (tc in test_cases) {
  cat('Text  :', tc$text, '\n')
  cat('Intent:', predict_intent_ml(tc$text), '\n')
  cat('ABSA  :\n')
  for (asp in tc$aspects)
    cat(sprintf('  %-12s -> %s\n', asp, predict_aspect(tc$text, asp)))
  cat('\n')
}

=== Full Enriched Output ===

Text  : The food was absolutely delicious but the service was shockingly rude. 
Intent: general 
ABSA  :
  food         -> positive
  service      -> negative

Text  : Great value for money! Arrived quickly and packaging was perfect. 
Intent: compliment 
ABSA  :
  price        -> positive
  shipping     -> positive
  packaging    -> positive

Text  : I wish the portion size was bigger. The taste is decent though. 
Intent: suggestion 
ABSA  :
  food         -> neutral
  service      -> negative

Text  : Does this come in a larger size? The quality looks great in the photos. 
Intent: general 
ABSA  :
  food         -> positive
  ambience     -> positive



## Files Saved

| File | Source | Description |
|---|---|---|
| `absa_model.rds` | MAMS-ACSA | glmnet ABSA classifier |
| `absa_vocab.rds` | MAMS-ACSA | Vocabulary for ABSA model |
| `intent_model.rds` | MAMS-ACSA (heuristic labels) | glmnet intent classifier |
| `intent_vocab.rds` | MAMS-ACSA | Vocabulary for intent model |

Add to `Dockerfile`:
```dockerfile
COPY absa_model.rds .
COPY absa_vocab.rds .
COPY intent_model.rds .
COPY intent_vocab.rds .
```

In [154]:
## ── Inference ──────────────────────────────────────────────────────────────────
## Edit `user_text` and `user_aspects` below, then run this cell.

user_text    <- "The pizza was amazing but the waiter was rude and it took forever to get the bill."
user_aspects <- c("food", "service", "staff")

# ── load helpers if running this cell standalone ─────────────────────────────
if (!exists('absa_model_loaded')) {
  library(tm); library(Matrix); library(glmnet)
  absa_model_loaded   <- readRDS('absa_model.rds')
  absa_vocab_loaded   <- readRDS('absa_vocab.rds')
  intent_model_loaded <- readRDS('intent_model.rds')
  intent_vocab_loaded <- readRDS('intent_vocab.rds')
  clean_text <- function(x) {
    corpus <- VCorpus(VectorSource(as.character(x)))
    corpus <- tm_map(corpus, content_transformer(tolower))
    corpus <- tm_map(corpus, removePunctuation)
    corpus <- tm_map(corpus, removeNumbers)
    corpus <- tm_map(corpus, removeWords, stopwords('en'))
    corpus <- tm_map(corpus, stripWhitespace)
    corpus
  }
}

# ── predict helpers ───────────────────────────────────────────────────────────
infer_sentiment <- function(text, aspect) {
  inp    <- paste(text, paste0('ASPECT_', toupper(gsub(' ', '_', aspect))))
  corpus <- clean_text(inp)
  dtm    <- DocumentTermMatrix(corpus, control = list(dictionary = absa_vocab_loaded))
  x      <- Matrix::sparseMatrix(i = dtm$i, j = dtm$j, x = as.numeric(dtm$v),
                                 dims = dim(dtm), dimnames = dimnames(dtm))
  as.character(predict(absa_model_loaded, newx = x, s = 'lambda.min', type = 'class'))
}

infer_intent <- function(text) {
  corpus <- clean_text(text)
  dtm    <- DocumentTermMatrix(corpus, control = list(dictionary = intent_vocab_loaded))
  x      <- Matrix::sparseMatrix(i = dtm$i, j = dtm$j, x = as.numeric(dtm$v),
                                 dims = dim(dtm), dimnames = dimnames(dtm))
  as.character(predict(intent_model_loaded, newx = x, s = 'lambda.min', type = 'class'))
}

# ── run inference ─────────────────────────────────────────────────────────────
intent_result <- infer_intent(user_text)
absa_results  <- data.frame(
  Aspect    = user_aspects,
  Sentiment = vapply(user_aspects, function(a) infer_sentiment(user_text, a), character(1)),
  stringsAsFactors = FALSE
)

polarity_emoji <- c(positive = '+', negative = '-', neutral = '~')
absa_results$Score <- polarity_emoji[absa_results$Sentiment]

# ── pretty output ─────────────────────────────────────────────────────────────
cat('Input text :', user_text, '\n')
cat('Intent     :', toupper(intent_result), '\n\n')
cat('Aspect-level sentiment:\n')
kable(absa_results[, c('Aspect', 'Sentiment', 'Score')], format = 'simple', row.names = FALSE)


Input text : The pizza was amazing but the waiter was rude and it took forever to get the bill. 
Intent     : GENERAL 

Aspect-level sentiment:




Aspect    Sentiment   Score 
--------  ----------  ------
food      neutral     ~     
service   neutral     ~     
staff     negative    -     